In [1]:
!pip install --upgrade peft torchao


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python.exe -m pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1


In [6]:
import os
import json
import time
import shutil
from os import mkdir

import torch
import torch.nn as nn
import kagglehub

from torch.amp import GradScaler, autocast
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import get_peft_model, LoraConfig
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from tqdm.notebook import tqdm

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cpu).


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [ ]:
# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# Global Configuration Dictionary
CONFIG = {
    "data_dir": "./data",
    "batch_size": 128,
    "num_workers": 2,
    "num_classes": 2,
    "image_size": 224,
    "lr": 1e-4,
    "epochs": 5,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "save_dir": "./saved_model",
    "model_name": "google/vit-base-patch16-224",
}

# Label mapping
LABEL_MAP = {0: "FAKE", 1: "REAL"}

# Method Definitions:

In [ ]:
def download_data(config):
    """Downloads the CIFAKE dataset and moves it to the local data directory."""
    target_dir = config["data_dir"]

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"Dataset already exists at {target_dir}")
        return

    print("Downloading dataset via kagglehub...")
    cache_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

    print(f"Moving dataset from cache to {target_dir}...")
    os.makedirs(os.path.dirname(target_dir), exist_ok=True)
    shutil.copytree(cache_path, target_dir, dirs_exist_ok=True)
    print("Dataset ready!")

def get_dataloaders(config):
    """Creates and returns training and testing dataloaders."""
    train_transform = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_transform = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dir = os.path.join(config["data_dir"], "train")
    test_dir = os.path.join(config["data_dir"], "test")

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=True
    )

    return train_loader, test_loader, train_dataset.class_to_idx

In [ ]:
def build_model(config, device):
    """Loads the ViT model, applies LoRA, and moves it to the target device."""
    print("Loading base ViT model...")
    model = ViTForImageClassification.from_pretrained(
        config["model_name"],
        num_labels=config["num_classes"],
        ignore_mismatched_sizes=True
    )

    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        target_modules=["query", "value"],
    )

    model = get_peft_model(model, lora_config)
    model = model.to(device)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"LoRA Applied | Trainable %: {100 * trainable_params / total_params:.2f}%")
    return model

In [ ]:
def evaluate_epoch(model, loader, criterion, device, epoch, epochs):
    """Validates the model at the end of an epoch."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    val_bar = tqdm(loader, desc=f" Validating Epoch {epoch}/{epochs}", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")

    return avg_loss, acc, f1

def train_model(model, train_loader, test_loader, config, device, class_to_idx, label_map):
    """Main training loop."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config["lr"])
    scaler = GradScaler(device.type)

    epochs = config["epochs"]
    print(f"\n Starting Training for {epochs} Epochs...")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        start_time = time.time()

        batch_bar = tqdm(train_loader, desc=f" Epoch {epoch}/{epochs} Training", leave=False)

        for images, labels in batch_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = total_loss / len(train_loader)
        epoch_time = time.time() - start_time

        val_loss, val_acc, val_f1 = evaluate_epoch(model, test_loader, criterion, device, epoch, epochs)

        print(f"Epoch [{epoch}/{epochs}] — {epoch_time:.1f}s")
        print(f"Train  → Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"Val    → Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    # Save Model
    os.makedirs(config["save_dir"], exist_ok=True)
    model.save_pretrained(config["save_dir"])

    config_path = os.path.join(config["save_dir"], "training_config.json")
    save_config = {**config, "label_map": label_map, "class_to_idx": class_to_idx}
    with open(config_path, "w") as f:
        json.dump(save_config, f, indent=4)

    print(f"\n Model and config saved to {config['save_dir']}")

def final_evaluate(model, test_loader, device):
    """Runs a full evaluation on the test set and prints metrics."""
    model.eval()
    all_preds, all_labels = [], []

    eval_bar = tqdm(test_loader, desc="Final Evaluation")

    with torch.no_grad():
        for images, labels in eval_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")
    f1 = f1_score(all_labels, all_preds, average="weighted")

    print("\n" + "=" * 60)
    print("FINAL EVALUATION REPORT")
    print("=" * 60)
    print(f" Accuracy  : {accuracy:.4f}  ({accuracy * 100:.2f}%)")
    print(f" Precision : {precision:.4f}")
    print(f" Recall    : {recall:.4f}")
    print(f" F1 Score  : {f1:.4f}")
    print("=" * 60)

    print("\n Per-Class Classification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=["FAKE", "REAL"]))

# Execution Steps:

In [ ]:
# 1. Download Dataset
download_data(CONFIG)

In [3]:
# 2. Get DataLoaders
train_loader, test_loader, class_to_idx = get_dataloaders(CONFIG)

NameError: name 'get_dataloaders' is not defined

In [31]:
# 3. Build Model
model = build_model(CONFIG, DEVICE)

Loading base ViT model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


LoRA Applied | Trainable %: 0.68%


In [ ]:
# 4. Train Model
train_model(model, train_loader, test_loader, CONFIG, DEVICE, class_to_idx, LABEL_MAP)


 Starting Training for 5 Epochs...


 Epoch 1/5 Training:   0%|          | 0/405 [00:00<?, ?it/s]

In [ ]:
# 5. Final Evaluation
final_evaluate(model, test_loader, DEVICE)